# Stochastic Calculus from Zero · Episode 14: The Itô isometry, and why trading gains are martingales

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Mean zero · 4. The isometry · 5. Trading gains · 6. Peeking · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 14,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

The right-end sum is $\tfrac12 W_T^2 + \tfrac12 \sum (\Delta W)^2$, which tends to $\tfrac12 W_T^2 + \tfrac12 T$.

In [ ]:
out["prev"] = {"answer": "1/2 W^2 + 1/2 T"}

## 3. Mean zero

An Itô integral is the limit of sums $\sum H_{t_i}\,\Delta W_i$ where $H_{t_i}$ is known at $t_i$ (**adapted**). Each term has mean zero: $E[H_{t_i}\Delta W_i] = E\big[H_{t_i}\,E[\Delta W_i \mid \mathcal{F}_{t_i}]\big] = 0$. So $E\big[\int_0^T H\,dW\big] = 0$, and more: $\int_0^t H\,dW$ is a martingale in $t$.

## 4. The isometry

Squaring the sum, the cross terms $E[H_i \Delta W_i\, H_j \Delta W_j]$ ($i < j$) vanish for the same reason, and each diagonal term is $E[H_i^2]\,\Delta t$:

$$E\Big[\Big(\int_0^T H\,dW\Big)^2\Big] = E\Big[\int_0^T H_t^2\,dt\Big] .$$

This is the **Itô isometry**. For $H = W$: $E\big[\int_0^T W_t^2\,dt\big] = \int_0^T t\,dt = T^2/2$.

In [ ]:
rng = np.random.default_rng(seed)
P, N_STEPS = 20_000, 500
_, W, dW = brownian_paths(P, N_STEPS, 1.0, rng)
I = (W[:, :-1] * dW).sum(axis=1)                         # left sums of W dW
out["isometry"] = {"paths": P, "mean": float(I.mean()), "mean_sq": float((I ** 2).mean()), "theory": 0.5,
                   "se": float((I ** 2).std() / math.sqrt(P))}
assert abs(out["isometry"]["mean"]) < 4 * float(I.std()) / math.sqrt(P)
assert abs(out["isometry"]["mean_sq"] - 0.5) < 3 * out["isometry"]["se"]
out["isometry"]

## 5. Trading gains

Think of $H$ as a position in an asset whose price moves like $W$, and $\int H\,dW$ as the trading gain. Any rule that decides $H$ from the past gives gains with mean zero. Example: "long one unit while $W > 0$, short one unit while $W < 0$". Its gains average zero, and by the isometry their variance is $E[\int_0^1 1\,dt] = 1$.

In [ ]:
H = np.where(W[:, :-1] > 0, 1.0, -1.0)
G = (H * dW).sum(axis=1)
out["sign"] = {"mean": float(G.mean()), "var": float(G.var()), "se": float(G.std() / math.sqrt(P))}
assert abs(out["sign"]["mean"]) < 4 * out["sign"]["se"] and abs(out["sign"]["var"] - 1) < 0.05
out["sign"]

## 6. Peeking

Now break the rule: choose the position *after* seeing the step, $H_i = \mathrm{sign}(\Delta W_i)$. Every step is then a gain of $|\Delta W_i|$, and the total averages $N\sqrt{2\Delta t/\pi} = \sqrt{2N/\pi}$, which grows without limit as trading gets more frequent (Episode 10's infinite length, cashed in). The only way to beat a fair game is to peek.

In [ ]:
peek = np.abs(dW).sum(axis=1)                             # sign(dW) * dW
out["peek"] = {"mean": float(peek.mean()), "theory": math.sqrt(2 * N_STEPS / math.pi), "steps": N_STEPS}
assert abs(out["peek"]["mean"] / out["peek"]["theory"] - 1) < 0.01
out["table"] = [{"strategy": "Hold W (the integral of W dW)", "mean": out["isometry"]["mean"]},
                {"strategy": "Long if W > 0, short if W < 0", "mean": out["sign"]["mean"]}]   # the peek is its own scene
out["peek"]

## 7. This episode's question

$\mathrm{Var}\big(\int_0^4 2\,dW\big) = \int_0^4 2^2\,dt = 16$.

In [ ]:
out["question"] = {"H": 2, "T": 4, "answer": 4 * 4}

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Two steps of length $\Delta t$: $H_0 = 1$, $H_1 = W_{\Delta t}$. Compute $E[(H_0\Delta W_0 + H_1 \Delta W_1)^2]$ directly and check it equals $E[H_0^2 + H_1^2]\,\Delta t$.

<details><summary>Answer</summary>

Cross term $2E[\Delta W_0 \cdot W_{\Delta t} \Delta W_1] = 0$ since $\Delta W_1$ is independent with mean 0. Diagonal: $\Delta t + E[W_{\Delta t}^2]\Delta t = \Delta t + \Delta t^2$, which is $E[H_0^2 + H_1^2]\Delta t$. ✓

</details>

**Exercise 2 (By hand).** Use the isometry to find $\mathrm{Var}\big(\int_0^T W\,dW\big)$, and check it against Episode 13's formula $\tfrac12 W_T^2 - \tfrac12 T$.

<details><summary>Answer</summary>

Isometry: $\int_0^T t\,dt = T^2/2$. Directly: $\mathrm{Var}(\tfrac12 W_T^2) = \tfrac14 \mathrm{Var}(W_T^2) = \tfrac14 \cdot 2T^2 = T^2/2$. ✓

</details>

**Exercise 3 (By hand).** Why does the peeking gain grow like $\sqrt{N}$ with the number of trades?

<details><summary>Answer</summary>

Each step earns $|\Delta W| \approx \sqrt{2\Delta t/\pi}$, and there are $N = T/\Delta t$ steps: $N\sqrt{2T/(\pi N)} = \sqrt{2TN/\pi}$.

</details>

**Exercise 4 (In code).** Check the isometry for $H = W^2$: $E[(\int_0^1 W^2 dW)^2] = \int_0^1 E[W_t^4]\,dt = \int_0^1 3t^2\,dt = 1$.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
J = (W[:, :-1] ** 2 * dW).sum(axis=1)
se = float((J ** 2).std() / math.sqrt(P))
assert abs((J ** 2).mean() - 1) < 4 * se
print(f"E[(integral of W^2 dW)^2] = {(J ** 2).mean():.3f} (theory 1, standard error {se:.3f})")

**Exercise 5 (In code).** A strategy that reacts to the previous step, $H_i = \mathrm{sign}(\Delta W_{i-1})$, uses only the past. Check its gains still average zero.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
Hm = np.concatenate([np.zeros((P, 1)), np.sign(dW[:, :-1])], axis=1)
Gm = (Hm * dW).sum(axis=1)
assert abs(Gm.mean()) < 4 * Gm.std() / math.sqrt(P)
print(f"momentum strategy: mean gain {Gm.mean():+.4f} (standard error {Gm.std() / math.sqrt(P):.4f})")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")